# FS-005 Phase 5 — Kaggle final GPU session (paper subset)

One unattended session that finishes ALL remaining generations:
1. MedGemma Track C **Condition-1 greedy** run (250 q, ~0.7 h) — first, so the greedy-decoding fix is verified cheaply.
2. Qwen2.5-VL-3B Track B (validation fit/assess + test).
3. Qwen2.5-VL-3B Track C (validation fit/assess + test + Condition-1 greedy).

Each job smoke-tests 3 questions first; a failing job is skipped (the others still run) and everything is zipped at the end. A time guard stops starting new stages after 10.5 h so the 12 h limit is never hit.

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**, Secret `HF_TOKEN` attached.
**Run it:** top bar -> **Save Version -> Save & Run All (Commit)**. Output: `fs005_generations_session2.zip` on the version's Output tab.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one GPU only
import subprocess, sys
REPO = '/kaggle/working/MedicalVQA'
OUT = '/kaggle/working/fs005_phase5_full'
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/Abishakm1507/MedicalVQA.git',REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'], check=True)
print(subprocess.run(['git','-C',REPO,'log','-1','--format=Running commit: %h %s'], capture_output=True, text=True).stdout)
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','pydantic>=2.6','transformers>=4.46','accelerate','bitsandbytes','safetensors','huggingface_hub','qwen-vl-utils'], check=True)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami
login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)   # token is never printed
print('HF account:', whoami()['name'])
import torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2."
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Remaining work (MedGemma Track B/C stages 1-3 were already generated in session 1).
JOBS = [
    ('medgemma-4b',   'C', ['test_c1']),
    ('qwen2.5-vl-3b', 'B', ['val_fit', 'val_assess', 'test']),
    ('qwen2.5-vl-3b', 'C', ['val_fit', 'val_assess', 'test', 'test_c1']),
]
MAX_HOURS = 10.5   # do not START a new stage after this many hours (Kaggle commit limit is 12 h)

In [ ]:
import subprocess, time, shutil
T0 = time.time()
def sh(args):
    t = time.time()
    print('\n$', ' '.join(args), flush=True)
    p = subprocess.Popen([sys.executable] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'it/s' in line or 'B/s' in line:      # skip download/progress bar noise
            continue
        print(line, end='', flush=True)
    p.wait()
    print(f'[exit {p.returncode}, {(time.time()-t)/60:.1f} min, session {(time.time()-T0)/3600:.2f} h]', flush=True)
    return p.returncode

def gen(model, track, split, limit=0):
    return sh(['scripts/fs005_phase5_full_run.py', 'generate', '--model', model, '--track', track,
               '--split', split, '--subset', 'paper', '--limit', str(limit), '--out-dir', OUT])

skipped = []
for model, track, splits in JOBS:
    # Fail fast on 3 questions (part of the subset, so reused, not wasted). A failing job is skipped.
    if gen(model, track, splits[0], limit=3) != 0:
        print(f'*** {model} track {track}: smoke generation FAILED; skipping this job.', flush=True)
        skipped.append((model, track))
        continue
    for split in splits:
        if (time.time() - T0) / 3600 > MAX_HOURS:
            print(f'*** time guard: not starting {model} {track} {split}', flush=True)
            skipped.append((model, track, split))
            continue
        gen(model, track, split)
print('SKIPPED:', skipped)
sh(['scripts/fs005_phase5_full_run.py', 'status', '--subset', 'paper', '--out-dir', OUT])
shutil.make_archive('/kaggle/working/fs005_generations_session2', 'zip', OUT)
print(os.listdir('/kaggle/working'))

## After this session (on your laptop, no GPU)
Download `fs005_generations_session2.zip` from the Output tab and tell Claude Code where it is. It will merge it with the session-1 files and run calibrate / evaluate / analyze and rebuild the paper.